# Parametric PINN frequency sweep (extended plots)
This notebook runs the sweep and makes multiple summary plots.

In [ ]:
import subprocess, shlex
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [ ]:
cmd = 'python parametric_pinn_frequency.py --freqs 10 20 30 40 50 60 --tmax 1.0 --npts 200 --adam_iter 150'
print(cmd)
# Uncomment next line to run
# subprocess.run(shlex.split(cmd), check=True, cwd='.')

In [ ]:
mat = loadmat('../Result/parametric_pinn_data.mat')
freq = mat['freq_hz'].squeeze()
time = mat['time'].squeeze()
x = mat['x']  # [n_freq, n_time, n_dof]
print('freq:', freq.shape, 'time:', time.shape, 'x:', x.shape)

## Plot 1: Peak response vs frequency

In [ ]:
peak = np.max(np.abs(x[:,:,0]), axis=1)
plt.figure(figsize=(6,4))
plt.plot(freq, peak, 'o-')
plt.xlabel('Input frequency [Hz]')
plt.ylabel('Peak |x1|')
plt.title('Frequency response (peak metric)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 2: RMS response vs frequency

In [ ]:
rms = np.sqrt(np.mean(x[:,:,0]**2, axis=1))
plt.figure(figsize=(6,4))
plt.plot(freq, rms, 's-')
plt.xlabel('Input frequency [Hz]')
plt.ylabel('RMS x1')
plt.title('Frequency response (RMS metric)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 3: Time history map across frequencies

In [ ]:
plt.figure(figsize=(8,4.5))
plt.pcolormesh(time, freq, x[:,:,0], shading='auto', cmap='viridis')
plt.xlabel('Time')
plt.ylabel('Input frequency [Hz]')
plt.title('x1(t) across frequency sweep')
plt.colorbar(label='x1')
plt.tight_layout()
plt.show()

## Plot 4: Example FFT spectra for selected frequencies

In [ ]:
dt = float(time[1]-time[0]) if len(time)>1 else 1.0
faxis = np.fft.rfftfreq(len(time), d=dt)
choose = [0, len(freq)//2, len(freq)-1]

plt.figure(figsize=(8,4.5))
for idx in choose:
    y = x[idx,:,0] - np.mean(x[idx,:,0])
    Y = np.abs(np.fft.rfft(y))
    plt.plot(faxis, Y, label=f'input {freq[idx]:.1f} Hz')
plt.xlabel('Output spectral frequency')
plt.ylabel('|FFT|')
plt.title('Output spectra at selected input frequencies')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Should you change excitation frequency and plot again?
**Definitely yes.** A sweep over many frequencies is the right way to reveal resonances and improve dispersion interpretation.